In [5]:
import sys; sys.path.append("..")
from src.generate_data import make_subscribers
subs = make_subscribers()

ImportError: cannot import name 'make_subscribers' from 'src.generate_data' (/Users/michellecarrizosa/Interviewing/ai-care-measurement/src/generate_data.py)

In [7]:
!grep -n "def " src/generate_data.py

220:def logistic(x):
228:def generate_patients(rng):
265:def simulate_churn(patients, adopter, engagement, rng):
316:def draw_intents(program, centered_risk, rng):
329:def draw_handle_minutes(intent, urgent, from_ai, rng):
336:def generate_contacts(patients, cancel, adopter, rng):
377:def assign_experiment(ai_contacts, rng):
395:def simulate_ai_conversations(c, rng):
464:def build_tickets(direct, conversations, rng):
502:def inject_data_quality_issues(conversations, rng):
521:def main():
611:def print_checks(patients, subscriptions, conversations, tickets, qa_reviews, exposures):


In [8]:
!python src/generate_data.py

Rows written:
  patients                20,000
  subscriptions           20,000
  conversations           64,163
  support_tickets         94,725
  qa_reviews               3,192
  experiment_exposures     4,711

Self-selection: AI chat use by risk quintile
               baseline_risk  used_ai_chat
risk_quintile                             
Q1 low                 0.058         0.234
Q2                     0.150         0.286
Q3                     0.240         0.333
Q4                     0.351         0.416
Q5 high                0.547         0.527

Naive comparison (biased on purpose): share who cancelled
used_ai_chat
False    0.399
True     0.246

Experiment split: {'v1': 2361, 'v2': 2350}


## 1.Check SRM formally (you just saw it's fine). sample ratio mismatch.
## 2.Define the primary metric, resolution rate, and the guardrails: escalation rate for symptom concerns, and urgent escalations.
## 3.Compare v1 and v2 on each, with confidence intervals.
## 4.Make the decision call.

In [9]:
from scipy.stats import chisquare

In [10]:
import pandas as pd

In [11]:
counts = [2361, 2350]               # v1, v2
expected = [sum(counts) / 2] * 2    # 50/50 design
stat, p = chisquare(counts, f_exp=expected)
print(f"chi2 = {stat:.3f}, p = {p:.3f}")

chi2 = 0.026, p = 0.873


# 1 The program and plan shares are close to the settings (55/30/15 and 60/25/15).
# 2 No conversation for any program happens before that program's launch date. Join convos to patients and launches to check.
# 3 Chat users have higher baseline_risk than non-users. That's the self-selection.
# 4 The naive cancellation comparison makes chat look great. The docstring says this overstates the effect, and figuring out by how much is the whole project.

In [13]:
import pandas as pd

raw = "data/raw/"
patients = pd.read_csv(raw + "patients.csv", parse_dates=["signup_date"])
subs = pd.read_csv(raw + "subscriptions.csv", parse_dates=["start_date", "cancel_date"])
convos = pd.read_csv(raw + "conversations.csv", parse_dates=["started_at"])
tickets = pd.read_csv(raw + "support_tickets.csv", parse_dates=["created_at"])
launches = pd.read_csv(raw + "ai_launches.csv", parse_dates=["launch_date"])

In [14]:
subs.head(10)

,patient_id,start_date,cancel_date
0,1,2025-03-26,2025-10-22
1,2,2024-10-12,NaT
2,3,2025-05-02,2025-06-01
3,4,2025-02-18,NaT
4,5,2024-03-13,NaT
5,6,2025-06-20,2025-07-20
6,7,2025-03-20,NaT
7,8,2025-03-31,NaT
8,9,2024-04-06,2024-10-03
9,10,2024-10-18,2024-11-17


In [16]:
convos.head(10)

,conversation_id,patient_id,started_at,model_version,intent,confidence,escalated,escalation_type,resolved,csat
0,1,5324,2024-04-01 01:37:00,v1,shipping,0.867,True,support_agent,True,NaN
1,2,9855,2024-04-01 07:57:00,v1,side_effects,0.645,True,clinician,True,NaN
2,3,2571,2024-04-01 08:25:00,v1,side_effects,0.685,False,NaN,True,5.0
3,4,17510,2024-04-01 08:35:00,v1,shipping,0.986,False,NaN,True,NaN
4,5,424,2024-04-01 09:06:00,v1,shipping,0.735,True,support_agent,True,4.0
5,6,8989,2024-04-01 09:19:00,v1,dosing_question,0.726,True,clinician,True,5.0
6,7,8989,2024-04-01 09:41:00,v1,side_effects,0.869,False,NaN,True,NaN
7,8,10185,2024-04-01 09:43:00,v1,dosing_question,0.621,False,NaN,False,NaN
8,9,10693,2024-04-01 10:18:00,v1,symptom_concern,0.427,True,urgent_clinical,True,3.0
9,10,19461,2024-04-01 10:45:00,v1,dosing_question,0.623,False,NaN,False,1.0


In [24]:
patients.head(5)

,patient_id,signup_date,program,plan,acquisition_channel,age,baseline_risk
0,1,2025-03-26,hair,monthly,referral,48,0.126
1,2,2024-10-12,bp,monthly,search,41,0.100
2,3,2025-05-02,weight,monthly,referral,46,0.335
3,4,2025-02-18,weight,monthly,organic,41,0.483
4,5,2024-03-13,hair,annual,search,44,0.456


In [25]:
launches.head(5)

,program,launch_date
0,weight,2024-04-01
1,hair,2024-10-01
2,bp,2025-03-01


In [17]:
print(patients["program"].value_counts(normalize=True).round(3))

program
weight    0.549
hair      0.302
bp        0.150
Name: proportion, dtype: float64


In [18]:
print(patients["plan"].value_counts(normalize=True).round(3))

plan
monthly      0.600
quarterly    0.252
annual       0.149
Name: proportion, dtype: float64


In [27]:
## no conversations before launch
check = (convos.merge(patients[["patient_id", "program"]], on="patient_id").merge(launches, on ="program"))

In [29]:
early = check[check["started_at"] < check["launch_date"]]
print(len(early))


0


In [30]:
print(check.groupby("program")["started_at"].min())

program
bp       2025-03-01 00:20:00
hair     2024-10-01 03:49:00
weight   2024-04-01 01:37:00
Name: started_at, dtype: datetime64[ns]


In [33]:
patients[["program"]]           

,program
0,hair
1,bp
2,weight
3,weight
4,hair
...,...
19995,bp
19996,bp
19997,hair
19998,weight
